# 💬 DeltAI Chat — Pretrain 33M

Chat interactif avec ton checkpoint pretrain (`best.pt`) sur Drive.

**Pré-requis :**
- `best.pt` et `tokenizer.json` doivent être sur Drive dans `MyDrive/DeltAI-EN/`
- Runtime → Change runtime type → **T4** (gratuit) ou L4

Le lien Gradio public s'affichera à la dernière cellule.

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
!pip install -q tokenizers gradio

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!ls -la /content/drive/MyDrive/DeltAI-EN/

Mounted at /content/drive
total 0


In [3]:
# Localise les fichiers (au cas où les chemins varient)
!find /content/drive/MyDrive -name 'tokenizer.json' 2>/dev/null
!find /content/drive/MyDrive -name 'best.pt' 2>/dev/null

/content/drive/MyDrive/delt/best.pt


In [20]:
# Copie tokenizer + checkpoint dans le runtime (adapte les chemins si besoin)
!mkdir -p data checkpoints
!cp '/content/drive/MyDrive/DeltAI-EN/tokenizer.json'       data/
!cp '/content/drive/MyDrive/DeltAI-EN/checkpoints/best.pt'  checkpoints/
!ls -la data/ checkpoints/   # vérifier tailles non-nulles

cp: cannot stat '/content/drive/MyDrive/DeltAI-EN/tokenizer.json': No such file or directory
cp: cannot stat '/content/drive/MyDrive/DeltAI-EN/checkpoints/best.pt': No such file or directory
checkpoints/:
total 103956
drwxr-xr-x 3 root root      4096 May 16 14:21 .
drwxr-xr-x 1 root root      4096 May 16 14:13 ..
-rw-r--r-- 1 root root 106431488 May 16 14:21 best.pt
drwxr-xr-x 2 root root      4096 May 16 14:21 .ipynb_checkpoints

data/:
total 1096
drwxr-xr-x 2 root root    4096 May 16 14:11 .
drwxr-xr-x 1 root root    4096 May 16 14:13 ..
-rw-r--r-- 1 root root 1113200 May 16 14:11 tokenizer.json


## 📝 Code source

In [24]:
%%writefile tokenizer.py
"""
tokenizer.py — BPE rapide via HuggingFace `tokenizers` (Rust)
Garde la même API que la version pure-Python (encode/decode/save/load).
Tokens spéciaux : <pad>, <unk>, <bos>, <eos>, <user>, <assistant>
"""
import os
from typing import List

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# ──────────────────────────────────────────────────────────────────────────────
# Tokens spéciaux (IDs stables, alignés avec le reste du projet)
# ──────────────────────────────────────────────────────────────────────────────
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>", "<user>", "<assistant>"]
PAD_ID, UNK_ID, BOS_ID, EOS_ID, USER_ID, ASST_ID = range(6)
N_SPECIAL = len(SPECIAL_TOKENS)


class BPETokenizer:
    """Wrapper autour de tokenizers.Tokenizer (BPE byte-level)."""

    def __init__(self, tk: Tokenizer | None = None):
        self.tk: Tokenizer | None = tk
        self.is_trained = tk is not None

    # ── Entraînement ──────────────────────────────────────────────────────────
    def train(self, texts: List[str], vocab_size: int = 8_000, verbose: bool = True):
        tk = Tokenizer(models.BPE(unk_token="<unk>"))
        tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tk.decoder = decoders.ByteLevel()

        trainer = trainers.BpeTrainer(
            vocab_size=vocab_size,
            special_tokens=SPECIAL_TOKENS,
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
            show_progress=verbose,
        )
        if verbose:
            print(f"[BPE] Entraînement Rust — vocab_size={vocab_size}")
        tk.train_from_iterator(texts, trainer=trainer)

        # Garantir que les IDs des specials sont 0..5 dans le bon ordre
        for expected_id, tok in enumerate(SPECIAL_TOKENS):
            real_id = tk.token_to_id(tok)
            assert real_id == expected_id, (
                f"ID inattendu pour {tok!r}: {real_id} (attendu {expected_id})"
            )

        self.tk = tk
        self.is_trained = True
        if verbose:
            print(f"[BPE] Terminé. Vocabulaire : {tk.get_vocab_size()} tokens")

    # ── Encodage / Décodage ───────────────────────────────────────────────────
    def encode(self, text: str) -> List[int]:
        return self.tk.encode(text, add_special_tokens=False).ids

    def decode(self, ids: List[int], skip_special: bool = True) -> str:
        if not skip_special:
            return self.tk.decode(ids, skip_special_tokens=False)
        # Remplace user/assistant par des marqueurs lisibles, ignore les autres specials
        out = []
        for i in ids:
            if i == USER_ID:
                out.append("\n[Utilisateur] ")
            elif i == ASST_ID:
                out.append("\n[Assistant] ")
            elif i in (PAD_ID, UNK_ID, BOS_ID, EOS_ID):
                continue
            else:
                out.append(self.tk.decode([i], skip_special_tokens=False))
        return "".join(out)

    # ── Sauvegarde / Chargement ───────────────────────────────────────────────
    def save(self, path: str):
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        self.tk.save(path)
        print(f"[BPE] Tokenizer sauvegardé → {path}")

    @classmethod
    def load(cls, path: str) -> "BPETokenizer":
        return cls(Tokenizer.from_file(path))

    @property
    def vocab_size(self) -> int:
        return self.tk.get_vocab_size()

    def __len__(self):
        return self.vocab_size


# ──────────────────────────────────────────────────────────────────────────────
# Test autonome
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    samples = [
        "Hello, how can I help you today?",
        "I need help understanding transformers.",
        "Sure! A transformer is a neural network architecture.",
        "Bonjour, comment puis-je vous aider ?",
        "Je voudrais entraîner un modèle de langage.",
    ] * 200

    tok = BPETokenizer()
    tok.train(samples, vocab_size=600, verbose=True)

    text = "Hello, how can I help you today?"
    ids  = tok.encode(text)
    dec  = tok.decode(ids, skip_special=False)
    print(f"\nOriginal : {text!r}")
    print(f"IDs      : {ids}")
    print(f"Décodé   : {dec!r}")


Overwriting tokenizer.py


In [25]:
%%writefile IA.py
"""
IA.py — MiniGPT ~35M paramètres
Architecture : d_model=512, n_head=8, n_layer=8, d_ff=2048, vocab=16000, block=512
Techniques   : Flash Attention, GELU, Weight Tying, Pre-LayerNorm, Dropout
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F


# ──────────────────────────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────────────────────────
@dataclass
class ModelConfig:
    vocab_size : int   = 16_000  # vocabulaire BPE plus riche
    block_size : int   = 512     # contexte 2× plus long
    d_model    : int   = 512     # dimension d'embedding
    n_head     : int   = 8       # nombre de têtes d'attention
    n_layer    : int   = 8       # nombre de blocs transformer
    d_ff       : int   = 2_048   # dimension de la couche feed-forward
    dropout    : float = 0.1     # dropout standard (overfit moins critique avec plus de data)
    bias       : bool  = False   # sans biais → plus rapide


# ──────────────────────────────────────────────────────────────────────────────
# Blocs du Transformer
# ──────────────────────────────────────────────────────────────────────────────
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention avec Flash Attention si disponible."""

    def __init__(self, cfg: ModelConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_head == 0
        self.h_dim  = cfg.d_model // cfg.n_head
        self.n_head = cfg.n_head
        self.dp     = cfg.dropout

        self.c_attn  = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.c_proj  = nn.Linear(cfg.d_model, cfg.d_model,     bias=cfg.bias)
        self.drop_a  = nn.Dropout(cfg.dropout)
        self.drop_r  = nn.Dropout(cfg.dropout)

        self.flash = hasattr(F, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer(
                "mask",
                torch.tril(torch.ones(cfg.block_size, cfg.block_size))
                     .view(1, 1, cfg.block_size, cfg.block_size),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)

        def reshape(t):
            return t.view(B, T, self.n_head, self.h_dim).transpose(1, 2)

        q, k, v = reshape(q), reshape(k), reshape(v)

        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v,
                dropout_p=self.dp if self.training else 0.0,
                is_causal=True,
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (self.h_dim ** -0.5)
            att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
            att = self.drop_a(F.softmax(att, dim=-1))
            y   = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop_r(self.c_proj(y))


class FeedForward(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.d_model)
        self.ffwd = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


# ──────────────────────────────────────────────────────────────────────────────
# Modèle complet
# ──────────────────────────────────────────────────────────────────────────────
class MiniGPT(nn.Module):
    def __init__(self, cfg: ModelConfig = None):
        super().__init__()
        cfg = cfg or ModelConfig()
        self.cfg = cfg

        self.wte    = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.wpe    = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop   = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f   = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Weight tying : la couche de sortie partage les poids de l'embedding
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        # Mise à l'échelle des projections résiduelles (important pour la stabilité)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"Contexte trop long ({T} > {self.cfg.block_size})"

        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x   = self.drop(self.wte(idx) + self.wpe(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))   # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=0.8,
                 top_k=50, top_p=0.95, eos_id=None):
        """Génère des tokens de façon auto-régressive."""
        for _ in range(max_new_tokens):
            ctx    = idx[:, -self.cfg.block_size:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / temperature

            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")

            if top_p < 1.0:
                sl, si = torch.sort(logits, descending=True)
                cum = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
                sl[cum - F.softmax(sl, dim=-1) > top_p] = float("-inf")
                logits.scatter_(1, si, sl)

            next_tok = torch.multinomial(F.softmax(logits, dim=-1), 1)
            idx = torch.cat([idx, next_tok], dim=1)
            if eos_id is not None and next_tok.item() == eos_id:
                break
        return idx

    def num_params(self):
        """Total de paramètres (embedding compté une seule fois, weight tying)."""
        return sum(p.numel() for p in self.parameters())

    def configure_optimizers(self, weight_decay, learning_rate, betas):
        """
        Sépare les paramètres : on applique le weight decay sur les tenseurs 2D+
        (poids des layers linéaires/emb), mais pas sur les tenseurs 1D
        (LayerNorm, bias). C'est crucial contre l'overfit.
        """
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]

        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas)
        return optimizer


# ──────────────────────────────────────────────────────────────────────────────
# Test rapide
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    cfg = ModelConfig()
    m   = MiniGPT(cfg)
    n   = m.num_params()
    print(f"Paramètres : {n:,}  (~{n/1e6:.1f}M)")
    assert 30_000_000 < n < 45_000_000, f"Attendu ~35M, obtenu {n}"
    x = torch.randint(0, cfg.vocab_size, (2, 64))
    _, loss = m(x, x)
    print(f"Forward OK — loss={loss.item():.4f}")

Overwriting IA.py


## 🤖 Chargement modèle + app Gradio

In [35]:
import torch, gradio as gr
from IA import MiniGPT, ModelConfig
from tokenizer import BPETokenizer, BOS_ID, EOS_ID, USER_ID, ASST_ID

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Chat] Device : {DEVICE}")

ck = torch.load("checkpoints/best.pt", map_location=DEVICE, weights_only=False)
cfg = ck["cfg"]
model = MiniGPT(cfg).to(DEVICE)
model.load_state_dict(ck["model"]); model.eval()
tok = BPETokenizer.load("data/tokenizer.json")

step = ck.get("step", "?")
val  = ck.get("val_loss", float("nan"))
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[Chat] Pretrain — step={step}  val_loss={val:.4f}  params={n_params/1e6:.1f}M")
print(f"[Chat] vocab={cfg.vocab_size}  d_model={cfg.d_model}  block={cfg.block_size}")


[Chat] Device : cuda
[Chat] Pretrain — step=15000  val_loss=2.0589  params=33.6M
[Chat] vocab=16000  d_model=512  block=512


In [36]:
@torch.no_grad()
def chat(prompt, temp, top_k, top_p, max_new):
    if not prompt.strip():
        return ""
    ids = [BOS_ID, USER_ID] + tok.encode(prompt) + [EOS_ID, ASST_ID]
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    if x.size(1) > cfg.block_size - int(max_new):
        x = x[:, -(cfg.block_size - int(max_new)):]
    out = model.generate(x, max_new_tokens=int(max_new),
                         temperature=float(temp),
                         top_k=int(top_k), top_p=float(top_p),
                         eos_id=EOS_ID)
    new_ids = out[0, x.size(1):].tolist()
    if new_ids and new_ids[-1] == EOS_ID: new_ids = new_ids[:-1]
    return tok.decode(new_ids, skip_special=True).strip()

with gr.Blocks(title="DeltAI Chat", theme=gr.themes.Soft(primary_hue="violet")) as demo:
    gr.HTML(f"""
    <div style='text-align:center; padding:14px'>
      <h1 style='margin:0; font-weight:700'>🤖 DeltAI</h1>
      <p style='opacity:.65; margin:6px'>
        33M params · step {step} · val {val:.3f} · <code>{DEVICE}</code>
      </p>
    </div>""")

    with gr.Row():
        with gr.Column(scale=3):
            out = gr.Textbox(lines=14, label="Réponse", interactive=False)
            with gr.Row():
                msg = gr.Textbox(placeholder="Pose ta question…", show_label=False,
                                 scale=8, autofocus=True)
                send = gr.Button("Envoyer", variant="primary", scale=1)
                clear = gr.Button("🗑️", scale=0)
        with gr.Column(scale=1):
            gr.Markdown("### 🎛️ Sampling")
            t  = gr.Slider(0.1, 1.5, value=0.8,  step=0.05, label="Temperature")
            tk = gr.Slider(1, 200,   value=50,   step=1,    label="Top-k")
            tp = gr.Slider(0.1, 1.0, value=0.95, step=0.01, label="Top-p")
            mx = gr.Slider(20, 400,  value=150,  step=10,   label="Tokens max")
            gr.Markdown("### 💡 Exemples")
            gr.Examples(
                examples=[
                    "Hello!",
                    "Write a Python function that adds two numbers",
                    "What is your favorite color?",
                    "Tell me a fact about space",
                    "Dear Boss,",
                    "Once upon a time,",
                ],
                inputs=msg,
            )

    send.click(chat, [msg, t, tk, tp, mx], out)
    msg.submit(chat, [msg, t, tk, tp, mx], out)
    clear.click(lambda: ("", ""), None, [msg, out])

demo.launch(share=True, debug=False)

/tmp/ipykernel_3514/3694958332.py:17: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="DeltAI Chat", theme=gr.themes.Soft(primary_hue="violet")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fa7f448ae42811feec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [34]:
!md5sum checkpoints/best.pt


4cfe39a94bf2deab0dc2a80d7f82dcad  checkpoints/best.pt
